In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [52]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7519.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [55]:
loader = PyPDFLoader("attention.pdf")
chunks = loader.load()

In [56]:
splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30
    )

In [57]:
docs = splitter.split_documents(chunks)

In [58]:
from langchain_community.vectorstores import MongoDBAtlasVectorSearch
from pymongo import MongoClient

In [ ]:
MONGODB_URI = "mongodb+srv://tanmay01bhatt:<db_password>@rag-cluster.bziqmbb.mongodb.net/?appName=RAG-Clustergodb.net/?appName=RAG-Cluster"

In [60]:
vector_store = MongoDBAtlasVectorSearch.from_connection_string(
  connection_string = MONGODB_URI,
  namespace = "langchain_db.test",
  embedding =  embeddings,
  index_name = "vector_index"
)

In [79]:
vector_store.add_documents(documents=docs)

[ObjectId('69e3744589e2faed3c21ed93'),
 ObjectId('69e3744589e2faed3c21ed94'),
 ObjectId('69e3744589e2faed3c21ed95'),
 ObjectId('69e3744589e2faed3c21ed96'),
 ObjectId('69e3744589e2faed3c21ed97'),
 ObjectId('69e3744589e2faed3c21ed98'),
 ObjectId('69e3744589e2faed3c21ed99'),
 ObjectId('69e3744589e2faed3c21ed9a'),
 ObjectId('69e3744589e2faed3c21ed9b'),
 ObjectId('69e3744589e2faed3c21ed9c'),
 ObjectId('69e3744589e2faed3c21ed9d'),
 ObjectId('69e3744589e2faed3c21ed9e'),
 ObjectId('69e3744589e2faed3c21ed9f'),
 ObjectId('69e3744589e2faed3c21eda0'),
 ObjectId('69e3744589e2faed3c21eda1'),
 ObjectId('69e3744589e2faed3c21eda2'),
 ObjectId('69e3744589e2faed3c21eda3'),
 ObjectId('69e3744589e2faed3c21eda4'),
 ObjectId('69e3744589e2faed3c21eda5'),
 ObjectId('69e3744589e2faed3c21eda6'),
 ObjectId('69e3744589e2faed3c21eda7'),
 ObjectId('69e3744589e2faed3c21eda8'),
 ObjectId('69e3744589e2faed3c21eda9'),
 ObjectId('69e3744589e2faed3c21edaa'),
 ObjectId('69e3744589e2faed3c21edab'),
 ObjectId('69e3744589e2fa

In [81]:
retriever = vector_store.as_retriever(
   search_type = "similarity",
   search_kwargs = { "k": 2}
)

In [70]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [71]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [72]:
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below.

    Context:
    {context}

    Question: {question}
    """
)

In [66]:
from langchain_core.runnables import RunnablePassthrough

In [82]:
chain = (
   { "context": retriever, "question": RunnablePassthrough()}
   | rag_prompt
   | llm
   | StrOutputParser()
)

In [83]:
question = "What are the key points discussed in the document?"

In [84]:
response = chain.invoke(question)

In [85]:
print(response)

Based on the provided context, the document appears to be a research paper or academic article, and the key points discussed are not explicitly stated. However, the page content mentions "Long Papers), pages 434–443. ACL, August 2013.\n12", which suggests that the document may be related to a conference or publication in the field of natural language processing or artificial intelligence, specifically the Association for Computational Linguistics (ACL) conference in 2013. 

Without more information or content from the document, it is difficult to determine the specific key points discussed. The embedding and metadata provided do not offer clear insights into the document's content.
